# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # A partir de la imagen:
        self.start = (0, 0)

        # Estanterías / paredes (bloquean el paso)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Celdas de piso resbaloso (panal amarillo)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales con su recompensa
        self.terminal_states = {
            (0, 5): 10.0,   # zona de entrega
            (2, 2): 2.0,    # estación de carga
            (3, 5): -10.0,  # peligro mortal
        }

        # Peligros no terminales
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _step(self, state, action):
        """Aplica una acción determinística; si golpea pared/borde se queda quieto."""
        next_state = (state[0] + action[0], state[1] + action[1])
        if self.is_valid_state(next_state):
            return next_state
        return state

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Las probabilidades dependen de si 'state' es resbaloso.
        Si golpea pared/borde, next_state = state.
        """
        # Los estados terminales no tienen dinámica (se quedan en sí mismos)
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_forward, p_side = 0.60, 0.20
        else:
            p_forward, p_side = 0.90, 0.05

        # Desviaciones: 90° a la izquierda / derecha respecto a la dirección
        # en la que "mira" el robot (rotación del vector de acción).
        dr, dc = action
        left_action = (-dc, dr)
        right_action = (dc, -dr)

        outcomes = {}
        for a, p in [(action, p_forward), (left_action, p_side), (right_action, p_side)]:
            ns = self._step(state, a)
            outcomes[ns] = outcomes.get(ns, 0.0) + p

        return list(outcomes.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    transitions = grid.get_transition_probs(state, action)
    return sum(p * V[s_next] for s_next, p in transitions)


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = {}
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            q_values = [
                expected_next_value(grid, s, a, V)
                for a in grid.actions
            ]
            V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, it

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        best_a, best_q = None, -float("inf")
        for a in grid.actions:
            q = expected_next_value(grid, s, a, V)
            if q > best_q:
                best_q, best_a = q, a
        policy[s] = best_a
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = {}
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            a = policy[s]
            V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    S = grid.states()
    non_terminal = [s for s in S if not grid.is_terminal(s)]

    # 1. política inicial arbitraria
    policy = {s: grid.actions[0] for s in non_terminal}
    history = []

    for it in range(1, max_iter + 1):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)

        # 3. mejora
        new_policy = policy_improvement(grid, V)

        n_changed = sum(1 for s in non_terminal if new_policy[s] != policy[s])
        history.append(n_changed)

        # 4. repetir hasta estabilidad
        if n_changed == 0:
            policy = new_policy
            break
        policy = new_policy

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.



## Respuestas — Parte 5

**1. Desde `START`, ¿el robot busca la entrega +10 o la carga +2?**
Con los parámetros base (`living_reward = -1.0`, `γ = 0.9`), la política óptima desde `START` es
`(0,0) → (0,1) → (0,2) → (1,2) → (2,2)`, es decir, el robot va a la **estación de carga +2**, no a la entrega +10. Aunque la recompensa terminal de la entrega es mayor, queda mucho más lejos y el costo acumulado de los pasos (‑1 cada uno, descontado) hace que el retorno esperado de ir a cargar sea mayor que el de cruzar todo el almacén (pasando además cerca de peligros) para llegar a la entrega.

**2. ¿Por qué una recompensa menor podría ser óptima?**
Porque lo que se maximiza no es la recompensa terminal por sí sola, sino el **retorno descontado total** $R(s_0)+\gamma R(s_1)+\gamma^2 R(s_2)+\dots$. Un objetivo cercano con recompensa menor puede superar a un objetivo lejano con recompensa mayor si el costo de los pasos adicionales (y el descuento $\gamma^{n}$ aplicado a una recompensa lejana) reduce más el valor que la diferencia entre +10 y +2.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
Las celdas resbalosas (1,2), (2,1) y (3,3) tienen mayor probabilidad de desviarse (0.20 a cada lado vs. 0.05 en piso normal). En la ruta óptima esto se nota en (1,2): el robot igual decide bajar hacia la carga porque, aunque puede desviarse, todas las desviaciones posibles desde esa celda siguen siendo relativamente seguras (no caen en un estado terminal negativo ni en un peligro). Si una celda resbalosa estuviera junto a un peligro mortal, la política evitaría pasar por ella o elegiría una acción "más segura" aunque no sea la más directa, precisamente porque la probabilidad de desviación es mayor allí.

**4. ¿Qué papel cumple el costo por paso -1?**
Actúa como una penalización por el tiempo/energía que toma cada paso. Es lo que crea la presión para preferir caminos **cortos**: entre dos metas alcanzables, el costo por paso favorece a la más cercana, y en general empuja al agente a no dar vueltas innecesarias. Cuanto más negativo es, más se prioriza la cercanía sobre la magnitud de la recompensa terminal (ver Experimento A y el Bonus).

**5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?**
Porque la dinámica depende del **tipo de piso** en el estado actual: piso normal usa (0.90, 0.05, 0.05) y piso resbaloso usa (0.60, 0.20, 0.20). Si se usara una única distribución fija para todos los estados, no se podría representar esta heterogeneidad del entorno; `get_transition_probs` necesita consultar `self.slippery_states` para decidir qué distribución aplicar en cada estado antes de calcular los efectos de golpear una pared o el borde del grid.



## Experimentos A, B, C

**Predicciones antes de ejecutar:**

- **Experimento A** (`living_reward = -0.1`): al bajar el costo por paso, caminar ya casi no cuesta nada, así que el robot debería preferir la **entrega +10** aunque quede lejos, en vez de conformarse con la carga +2. Predicción: la política desde `START` cambia y el robot va hacia (0,5).
- **Experimento B** (piso resbaloso 0.40 / 0.30 / 0.30): al aumentar mucho el ruido en las celdas resbalosas, cruzar por ellas se vuelve más riesgoso (más probabilidad de desviarse hacia un peligro o de "perder" pasos). Predicción: el valor de los estados que dependen de pasar por celdas resbalosas baja, pero como ambas rutas (a carga y a entrega) usan celdas resbalosas, es probable que la meta elegida **no cambie** (sigue yendo a la carga +2), aunque el valor esperado general disminuya.
- **Experimento C** (`gamma = 0.99`): con un factor de descuento más alto, el agente es más "paciente" y las recompensas lejanas pierden menos valor por el descuento. Predicción: la política favorecerá más la **entrega +10**, igual que en el Experimento A.


In [7]:
# ============================================================
# Experimentos A, B, C y Bonus
# ============================================================

def trace_path(policy, grid, start=None, max_steps=20):
    """Sigue la dirección 'deseada' de la política (trayectoria determinista,
    ignorando el ruido de transición) solo para fines ilustrativos."""
    s = start if start is not None else grid.start
    path = [s]
    for _ in range(max_steps):
        if grid.is_terminal(s):
            break
        a = policy[s]
        ns_ = (s[0] + a[0], s[1] + a[1])
        if not grid.is_valid_state(ns_):
            ns_ = s
        s = ns_
        path.append(s)
    return path


# ---------- Experimento A: living_reward = -0.1 ----------
grid_A = WarehouseMDP()
grid_A.living_reward = -0.1

V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)

print("=== EXPERIMENTO A (living_reward = -0.1) ===")
print("Iteraciones:", n_A)
print_policy(grid_A, pi_A)
print("Trayectoria desde START:", trace_path(pi_A, grid_A))
print(f"V(START) = {V_A[grid_A.start]:+.3f}  (baseline: {V_vi[grid.start]:+.3f})")


# ---------- Experimento B: piso resbaloso más resbaloso (0.40 / 0.30 / 0.30) ----------
class WarehouseMDP_Slippery40(WarehouseMDP):
    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]
        if state in self.slippery_states:
            p_forward, p_side = 0.40, 0.30
        else:
            p_forward, p_side = 0.90, 0.05

        dr, dc = action
        left_action = (-dc, dr)
        right_action = (dc, -dr)

        outcomes = {}
        for a, p in [(action, p_forward), (left_action, p_side), (right_action, p_side)]:
            ns_ = self._step(state, a)
            outcomes[ns_] = outcomes.get(ns_, 0.0) + p
        return list(outcomes.items())


grid_B = WarehouseMDP_Slippery40()
V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)

print("\n=== EXPERIMENTO B (piso resbaloso: 0.40/0.30/0.30) ===")
print("Iteraciones:", n_B)
print_policy(grid_B, pi_B)
print("Trayectoria desde START:", trace_path(pi_B, grid_B))
print(f"V(START) = {V_B[grid_B.start]:+.3f}  (baseline: {V_vi[grid.start]:+.3f})")


# ---------- Experimento C: gamma = 0.99 ----------
grid_C = WarehouseMDP()
grid_C.gamma = 0.99

V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)

print("\n=== EXPERIMENTO C (gamma = 0.99) ===")
print("Iteraciones:", n_C)
print_policy(grid_C, pi_C)
print("Trayectoria desde START:", trace_path(pi_C, grid_C))
print(f"V(START) = {V_C[grid_C.start]:+.3f}  (baseline: {V_vi[grid.start]:+.3f})")


=== EXPERIMENTO A (living_reward = -0.1) ===
Iteraciones: 26
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Trayectoria desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
V(START) = +1.649  (baseline: -2.575)

=== EXPERIMENTO B (piso resbaloso: 0.40/0.30/0.30) ===
Iteraciones: 22
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Trayectoria desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
V(START) = -2.706  (baseline: -2.575)

=== EXPERIMENTO C (gamma = 0.99) ===
Iteraciones: 24
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
Trayectoria desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), 


**Resultados observados:**

- **Experimento A** confirma la predicción: con `living_reward = -0.1` la política desde `START` cambia y el robot va hacia la **entrega +10**, incluso pasando cerca del peligro (1,4) `-3`, porque el costo de caminar ya casi no penaliza la ruta larga.
- **Experimento B** también confirma la predicción: aunque el valor general baja (más ruido = más riesgo), la meta elegida desde `START` **sigue siendo la carga +2** — la ruta corta compensa el riesgo adicional del piso resbaloso más inestable.
- **Experimento C** confirma la predicción: con `gamma = 0.99` el agente valora mucho más la recompensa `+10` lejana y la política desde `START` cambia hacia la **entrega +10**, igual que en el Experimento A pero por una razón distinta (más paciencia en vez de menor costo por paso).



### Bonus — umbral de `living_reward`

Buscamos, manteniendo `gamma = 0.9` y el resto de parámetros base, el valor aproximado de `living_reward` en el que la política óptima desde `START` deja de ir a la carga `+2` y empieza a ir a la entrega `+10`.


In [8]:
# ============================================================
# Bonus: umbral de living_reward donde cambia la política desde START
# ============================================================

def goal_from_start(policy, grid, max_steps=20):
    s = grid.start
    for _ in range(max_steps):
        if grid.is_terminal(s):
            return s
        a = policy[s]
        ns_ = (s[0] + a[0], s[1] + a[1])
        if not grid.is_valid_state(ns_):
            ns_ = s
        s = ns_
    return s


results = []
for lr in np.arange(-1.00, -0.09, 0.01):
    g = WarehouseMDP()
    g.living_reward = round(lr, 3)
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    goal = goal_from_start(pi, g)
    results.append((round(lr, 3), goal))

# Encuentra el punto donde cambia de CARGA (2,2) a ENTREGA (0,5)
threshold = None
for (lr_prev, g_prev), (lr_next, g_next) in zip(results, results[1:]):
    if g_prev != g_next:
        threshold = (lr_prev, lr_next)
        break

print("Meta alcanzada desde START para distintos living_reward (muestra):")
for lr, g in results[::10]:
    nombre = "ENTREGA +10" if g == (0, 5) else ("CARGA +2" if g == (2, 2) else str(g))
    print(f"  living_reward = {lr:+.2f} -> {nombre}")

print(f"\nEl cambio de política ocurre entre living_reward = {threshold[0]} y {threshold[1]}")
print("=> Umbral aproximado: living_reward ≈ -0.80")
print("   Para living_reward < -0.80 (más negativo/costoso), el robot prefiere la CARGA +2 (más cercana).")
print("   Para living_reward > -0.80 (más barato moverse), el robot prefiere ir por la ENTREGA +10 (más lejana).")


Meta alcanzada desde START para distintos living_reward (muestra):
  living_reward = -1.00 -> CARGA +2
  living_reward = -0.90 -> CARGA +2
  living_reward = -0.80 -> CARGA +2
  living_reward = -0.70 -> ENTREGA +10
  living_reward = -0.60 -> ENTREGA +10
  living_reward = -0.50 -> ENTREGA +10
  living_reward = -0.40 -> ENTREGA +10
  living_reward = -0.30 -> ENTREGA +10
  living_reward = -0.20 -> ENTREGA +10
  living_reward = -0.10 -> ENTREGA +10

El cambio de política ocurre entre living_reward = -0.8 y -0.79
=> Umbral aproximado: living_reward ≈ -0.80
   Para living_reward < -0.80 (más negativo/costoso), el robot prefiere la CARGA +2 (más cercana).
   Para living_reward > -0.80 (más barato moverse), el robot prefiere ir por la ENTREGA +10 (más lejana).



**Conclusión del bonus:** el cambio de política ocurre alrededor de `living_reward ≈ -0.80`.

- Para `living_reward` más negativo que ese umbral (caminar es "caro"), el robot prefiere la **carga +2**, porque el ahorro de pasos pesa más que la diferencia de recompensa terminal.
- Para `living_reward` menos negativo que ese umbral (caminar es "barato"), el robot prefiere arriesgarse a cruzar todo el almacén hasta la **entrega +10**, ya que el costo del camino largo deja de ser decisivo frente a la recompensa terminal mayor.

Esto es coherente con la intuición de las preguntas 2 y 4: el costo por paso es, junto con `gamma`, el parámetro que determina el balance entre "recompensa cercana y modesta" vs. "recompensa lejana y grande".
